In [ ]:
!pip install transformers datasets torch scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install transformers datasets -q

In [ ]:
# ✅ STEP 2: Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# ✅ STEP 3: Load and clean dataset
df = pd.read_csv("/content/Reddit_Data.csv")  # Path to your file
df = df[['clean_comment', 'category']].dropna()

# ✅ STEP 4: Train/test split
X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42)

# ✅ STEP 5: TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# ✅ STEP 6: Train Logistic Regression model
model = LogisticRegression()
model.fit(X_train_vec, y_train)

# ✅ STEP 7: Evaluation
y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))

# ✅ STEP 8: Inference function
def predict_sentiment(text):
    vec = vectorizer.transform([text])
    return model.predict(vec)[0]

# Example:
print("Predicted Sentiment:", predict_sentiment("Buddhism is very peaceful and calming"))

              precision    recall  f1-score   support

          -1       0.89      0.69      0.78      1597
           0       0.87      0.97      0.91      2654
           1       0.89      0.90      0.89      3179

    accuracy                           0.88      7430
   macro avg       0.88      0.85      0.86      7430
weighted avg       0.88      0.88      0.88      7430

Predicted Sentiment: 1


In [ ]:
# ✅ Step 2: Imports
import pandas as pd
import numpy as np
import torch
import os

from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ✅ Step 3: Disable WandB (optional, prevents login issues)
os.environ["WANDB_DISABLED"] = "true"

# ✅ Step 4: Load your dataset
df = pd.read_csv("/content/Reddit_Data.csv")  # Use your CSV path
df = df[['clean_comment', 'category']].dropna()

# ✅ Step 5: Label mapping for 3-class classification
label_map = {-1: 0, 0: 1, 1: 2}
reverse_map = {0: -1, 1: 0, 2: 1}
df['label'] = df['category'].map(label_map)

# ✅ Step 6: Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df[['clean_comment', 'label']])
dataset = dataset.train_test_split(test_size=0.2)

# ✅ Step 7: Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(example["clean_comment"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize, batched=True)

# ✅ Step 8: Load BERT model with 3 output labels
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)

# ✅ Step 9: Define evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# ✅ Step 10: Training arguments
training_args = TrainingArguments(
    output_dir="./bert_results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# ✅ Step 11: Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# ✅ Step 12: Train the model
trainer.train()

# ✅ Step 13: Predict function
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return reverse_map[predicted_class]

# ✅ Step 14: Test prediction
sample = "I love the peaceful teachings of Buddhism."
print("Predicted Sentiment:", predict_sentiment(sample))  # Should print -1, 0, or 1

Map:   0%|          | 0/29719 [00:00<?, ? examples/s]

Map:   0%|          | 0/7430 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-14-12e7be81b013>:61: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.329100,0.308466,0.916016,0.916684,0.917860,0.916016
2,0.199800,0.250458,0.937281,0.937434,0.937636,0.937281
3,0.109400,0.279778,0.944145,0.944129,0.944152,0.944145


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)